# MFCC feature space in 3-D

Compresses every training window from **100 frames × 13 coefficients = 1,300 dimensions**
to 3, then draws a Plotly 3-D scatter so you can see which *groups* the features form
per label.

**Prerequisites**

```bash
uv sync --extra notebook      # jupyterlab, plotly, scikit-learn
uv run ultron features        # → build/features.npz (1,638 windows right now)
```

**What to look for**

* tight blobs per label = the model has something learnable to separate;
* `la` overlapping `lb` = the `luz alta` / `luz baja` confusion we are chasing;
* the `noise` blob smeared across everything = why ambient audio gets misread as a command;
* hold-out points (diamonds) landing *inside* a training blob = the model has seen that
  region; landing outside = genuine extrapolation, and that is where recall drops.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

pio.renderers.default = "plotly_mimetype"  # works in Jupyter and in nbconvert

def repo_root() -> Path:
    """Works from the repo root or from notebooks/ (nbconvert runs in the latter)."""
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").is_file():
            return cand
    return here

ROOT = repo_root()
ARCHIVE = ROOT / "build/features.npz"
BUILD = ROOT / "build"
assert ARCHIVE.is_file(), f"run `uv run ultron features` first (looked in {ROOT / 'build'})"
print("repo root:", ROOT)

arch = np.load(ARCHIVE, allow_pickle=True)
X_raw = arch["features"]        # (N, 100, 13) float32 — frames x coefficients
labels = arch["labels"]         # (N,)
paths = arch["paths"]           # (N,)
print(f"windows: {X_raw.shape}  dtype={X_raw.dtype}")
print("labels:", {l: int((labels == l).sum()) for l in sorted(set(labels))})

# Hold-out windows (never in the archive) — loaded once, used by both overlays.
from ultron.features import extract_holdout

try:
    ho = extract_holdout(dataset_root=str(ROOT / "dataset"), labels=set(labels))
    print("hold-out windows:", dict(zip(*np.unique(ho["labels"], return_counts=True))))
except Exception as exc:  # noqa: BLE001 — notebook still runs without a hold-out
    ho = None
    print("hold-out skipped:", exc)

repo root: /Users/franzemil/orca/workspaces/ultron/t10-host-infer
windows: (1638, 100, 13)  dtype=float32
labels: {np.str_('la'): 738, np.str_('lb'): 495, np.str_('noise'): 200, np.str_('wake'): 205}
hold-out windows: {np.str_('la'): np.int64(2), np.str_('lb'): np.int64(7), np.str_('noise'): np.int64(2), np.str_('wake'): np.int64(2)}


## 1. Flatten to a vector

Each window starts as `(100, 13)`. Three ways to make it a single vector:

| `POOL` | dims | keeps |
|---|---|---|
| `flatten` (default) | 1300 | everything, including *where* in time each pattern is |
| `mean` | 13 | only the average spectrum — loses timing, which is what `la` vs `lb` needs |
| `meanstd` | 26 | average + variability |

Use `flatten` to see the real geometry; `mean` is a good sanity check for the
shift/padding problem (if the offset changes the mean, the label clusters smear).

In [2]:
POOL = "flatten"  # "flatten" | "mean" | "meanstd"

if POOL == "flatten":
    X = X_raw.reshape(len(X_raw), -1)
elif POOL == "mean":
    X = X_raw.mean(axis=1)
elif POOL == "meanstd":
    X = np.concatenate([X_raw.mean(axis=1), X_raw.std(axis=1)], axis=1)
else:
    raise ValueError(POOL)

X = X.astype(np.float32)
print(f"pooled: {X.shape}")

# source split, for marker shapes: originals vs augmented (vs synthetic)
def _source(p):
    p = str(p)
    for name in ("originals", "augmented", "synthetic"):
        if p.startswith(name):
            return name
    return "other"

sources = np.array([_source(p) for p in paths])
print({s: int((sources == s).sum()) for s in sorted(set(sources))})

# Hold-out pooled the same way, so both overlays use an identical representation.
H = None
if ho is not None and len(ho["features"]):
    if POOL == "flatten":
        H = ho["features"].reshape(len(ho["features"]), -1).astype(np.float32)
    elif POOL == "mean":
        H = ho["features"].mean(axis=1)
    else:
        H = np.concatenate([ho["features"].mean(axis=1), ho["features"].std(axis=1)], axis=1)
    print(f"hold-out pooled: {H.shape}")

pooled: (1638, 1300)
{np.str_('augmented'): 1252, np.str_('originals'): 386}
hold-out pooled: (13, 1300)


## 2. Reduce to 3 dimensions

* `PCA` — linear, deterministic, fast; the 3 axes are the directions of greatest variance.
* `t-SNE` — nonlinear; better local clusters, but distances between clusters mean nothing
  and the picture changes with `perplexity`.

We fit a 30-component PCA anyway (for the explained-variance plot in §4) and use its
first three components as the projection.

In [3]:
METHOD = "pca"  # "pca" | "tsne"

scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

pca30 = PCA(n_components=min(30, Xs.shape[1]), random_state=42).fit(Xs)
print(f"variance explained by 3 components: {pca30.explained_variance_ratio_[:3].sum():.1%}")
print(f"variance explained by 30 components: {pca30.explained_variance_ratio_.sum():.1%}")

if METHOD == "pca":
    Z = pca30.transform(Xs)[:, :3]
    axis_names = [f"PC{i + 1}" for i in range(3)]
else:
    Z = TSNE(n_components=3, perplexity=40, init="pca", random_state=42).fit_transform(Xs)
    axis_names = ["tSNE1", "tSNE2", "tSNE3"]

print(f"projection: {Z.shape} via {METHOD}")

variance explained by 3 components: 26.5%
variance explained by 30 components: 61.1%
projection: (1638, 3) via pca


## 3. The 3-D picture

Colour = label, shape = source (circle = your recording, diamond = augmentation).
Hover for the source file. Rotate it — the interesting overlaps are rarely frontal.

In [4]:
df = pd.DataFrame({
    axis_names[0]: Z[:, 0], axis_names[1]: Z[:, 1], axis_names[2]: Z[:, 2],
    "label": labels, "source": sources, "path": paths,
})

fig = go.Figure()
for label in sorted(df["label"].unique()):
    for source in sorted(df["source"].unique()):
        sub = df[(df.label == label) & (df.source == source)]
        if sub.empty:
            continue
        fig.add_trace(go.Scatter3d(
            x=sub[axis_names[0]], y=sub[axis_names[1]], z=sub[axis_names[2]],
            mode="markers",
            name=f"{label} ({source})",
            marker=dict(size=2.5, symbol="circle" if source == "originals" else "diamond", opacity=0.65),
            text=sub["path"], hovertemplate="%{text}<extra></extra>",
        ))
fig.update_layout(
    height=760,
    title=f"MFCC windows ({METHOD.upper()}, POOL={POOL}) — {len(df)} windows",
    legend=dict(itemsizing="constant"),
    scene=dict(xaxis_title=axis_names[0], yaxis_title=axis_names[1], zaxis_title=axis_names[2]),
)
BUILD.mkdir(exist_ok=True)
fig.write_html(BUILD / "mfcc_3d.html", include_plotlyjs="cdn")
print("saved:", BUILD / "mfcc_3d.html")
fig.show()

saved: /Users/franzemil/orca/workspaces/ultron/t10-host-infer/build/mfcc_3d.html


## 3a. The same thing in 2-D

Easier to read than a rotating cube, and it can carry **1σ ellipses** per label: where two
ellipses overlap, the model cannot separate those labels — that is the `la` ↔ `lb` problem
in one picture. The axes are the first two components of the projection chosen in §2
(PC1/PC2 for PCA, a 2-D t-SNE when `METHOD="tsne"`).

In [5]:
def ellipse_xy(points: np.ndarray, n_std: float = 1.0, n: int = 120):
    """1-sigma covariance ellipse around 2-D points (eigen-decomposition)."""
    if len(points) < 3:
        return None, None
    cov = np.cov(points.T)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = np.maximum(vals[order], 0.0), vecs[:, order]
    theta = np.linspace(0, 2 * np.pi, n)
    circle = np.vstack([np.cos(theta), np.sin(theta)])
    ell = vecs @ np.diag(n_std * np.sqrt(vals)) @ circle
    return ell[0] + points[:, 0].mean(), ell[1] + points[:, 1].mean()


if METHOD == "pca":
    Z2, names2 = pca30.transform(Xs)[:, :2], [axis_names[0], axis_names[1]]
else:
    Z2 = TSNE(n_components=2, perplexity=40, init="pca", random_state=42).fit_transform(Xs)
    names2 = ["tSNE1", "tSNE2"]

PALETTE = {"la": "#636EFA", "lb": "#EF553B", "noise": "#00CC96", "wake": "#AB63FA"}

fig_2d = go.Figure()
for label in sorted(set(labels)):
    pts = Z2[labels == label]
    fig_2d.add_trace(go.Scatter(
        x=pts[:, 0], y=pts[:, 1], mode="markers", name=f"{label} ({len(pts)})",
        marker=dict(size=3, opacity=0.35, color=PALETTE.get(label)),
        text=paths[labels == label], hovertemplate="%{text}<extra></extra>",
    ))
    ex, ey = ellipse_xy(pts)
    if ex is not None:
        fig_2d.add_trace(go.Scatter(
            x=ex, y=ey, mode="lines", name=f"{label} 1σ",
            line=dict(color=PALETTE.get(label), width=2, dash="dash"),
            hoverinfo="skip", showlegend=False,
        ))

fig_2d.update_layout(
    height=620, title=f"MFCC windows in 2-D ({METHOD.upper()}, POOL={POOL}) — dashed = 1σ ellipse",
    xaxis_title=names2[0], yaxis_title=names2[1],
    legend=dict(itemsizing="constant"),
)
fig_2d.write_html(BUILD / "mfcc_2d.html", include_plotlyjs="cdn")
print("saved:", BUILD / "mfcc_2d.html")
fig_2d.show()

saved: /Users/franzemil/orca/workspaces/ultron/t10-host-infer/build/mfcc_2d.html


### 2-D, with the hold-out on top

Unseen clips drawn as black diamonds over the same ellipses: inside a label's ellipse means
the model has seen that region of feature space; outside means genuine extrapolation.

In [6]:
if ho is not None and len(ho["features"]) and METHOD == "pca":
    HZ2 = pca30.transform(scaler.transform(H))[:, :2]
    fig_2d_ho = go.Figure()
    for trace in fig_2d.data:  # reuse the cloud + ellipses
        fig_2d_ho.add_trace(trace)
    fig_2d_ho.add_trace(go.Scatter(
        x=HZ2[:, 0], y=HZ2[:, 1], mode="markers+text", name="hold-out",
        marker=dict(size=9, symbol="diamond", color="black", line=dict(width=1, color="white")),
        text=[f"{lbl}" for lbl in ho["labels"]],
        textposition="top center",
        hovertext=list(ho["paths"]), hovertemplate="%{hovertext}<extra></extra>",
    ))
    fig_2d_ho.update_layout(
        height=620, title=f"2-D + hold-out ({len(ho['labels'])} unseen windows)",
        xaxis_title=names2[0], yaxis_title=names2[1],
    )
    fig_2d_ho.write_html(BUILD / "mfcc_2d_holdout.html", include_plotlyjs="cdn")
    print("saved:", BUILD / "mfcc_2d_holdout.html")
    fig_2d_ho.show()
elif METHOD != "pca":
    print("2-D hold-out overlay needs METHOD='pca' (t-SNE cannot project new points)")

saved: /Users/franzemil/orca/workspaces/ultron/t10-host-infer/build/mfcc_2d_holdout.html


## 3b. Does the picture mean anything?

Three cheap numbers, so the plot is not the only evidence:

* **silhouette** (in the 3-D projection) — how well-separated the labels are, `-1…1`;
* **centroid distance matrix** — which two labels sit closest, i.e. which pair the model will confuse;
* **nearest-centroid accuracy** — how far a trivial classifier gets in this space; if it is high,
  the geometry is fine and the failure is elsewhere (padding, offsets, threshold).

In [7]:
from sklearn.metrics import silhouette_score

proj = pd.DataFrame(Z, columns=["c0", "c1", "c2"])
proj["label"] = labels

sil = silhouette_score(Z, labels) if len(set(labels)) > 1 else float("nan")
print(f"silhouette (3-D projection, cosine): {sil:.3f}")

centroids = proj.groupby("label")[["c0", "c1", "c2"]].mean()
labels_u = list(centroids.index)
dist = pd.DataFrame(
    [[np.linalg.norm(centroids.loc[a] - centroids.loc[b]) for b in labels_u] for a in labels_u],
    index=labels_u, columns=labels_u,
).round(2)
print("\ncentroid distances (3-D):")
print(dist)

# nearest-centroid accuracy: how separable is the geometry, ignoring everything else
cent = {l: centroids.loc[l].to_numpy() for l in labels_u}
pred = np.array([
    min(labels_u, key=lambda l: np.linalg.norm(row - cent[l])) for row in Z
])
acc = float((pred == labels).mean())
print(f"\nnearest-centroid accuracy in 3-D: {acc:.1%}")
conf = pd.crosstab(pd.Series(labels, name="true"), pd.Series(pred, name="pred"))
print("\nconfusion (true x pred):")
print(conf)

# ── the same check in the FULL space (the 3-D view keeps only 26% of variance) ──
rng = np.random.default_rng(0)
idx = rng.choice(len(Xs), min(600, len(Xs)), replace=False)
print(f"\nsilhouette (full {Xs.shape[1]}-D, random 600): {silhouette_score(Xs[idx], labels[idx]):.3f}")

cent_full = {l: Xs[labels == l].mean(axis=0) for l in sorted(set(labels))}
labels_u2 = list(cent_full)
dist_full = pd.DataFrame(
    [[np.linalg.norm(cent_full[a] - cent_full[b]) for b in labels_u2] for a in labels_u2],
    index=labels_u2, columns=labels_u2,
).round(1)
print("centroid distances (full space):")
print(dist_full)

pair = dist_full.where(~np.eye(len(labels_u2), dtype=bool)).stack().idxmin()
print(f"closest pair: {pair[0]} <-> {pair[1]} at {dist_full.loc[pair]:.1f}")

mask = np.isin(labels, ["la", "lb"])
if mask.sum() > 2:
    sub, suby = Xs[mask], labels[mask]
    print(f"\nla vs lb only: {mask.sum()} windows, silhouette {silhouette_score(sub, suby):.3f}")
    c = {l: Xs[labels == l].mean(axis=0) for l in ("la", "lb")}
    gap = np.linalg.norm(c["la"] - c["lb"])
    within = float(np.mean([np.linalg.norm(r - c[l]) for r, l in zip(sub, suby)]))
    print(f"la-lb centroid gap {gap:.1f} vs within-class spread {within:.1f} (ratio {gap / within:.2f})")
    print("  ratio < 1 → the two labels overlap more than they separate")

silhouette (3-D projection, cosine): 0.073

centroid distances (3-D):
              la         lb      noise   wake
la      0.000000   2.680000  28.790001  26.75
lb      2.680000   0.000000  26.389999  24.17
noise  28.790001  26.389999   0.000000  11.65
wake   26.750000  24.170000  11.650000   0.00

nearest-centroid accuracy in 3-D: 61.4%

confusion (true x pred):
pred    la   lb  noise  wake
true                        
la     522  127      9    80
lb     280  134     25    56
noise    5    9    160    26
wake     0    5     10   190

silhouette (full 1300-D, random 600): 0.031
centroid distances (full space):
              la    lb      noise       wake
la      0.000000   8.6  30.700001  29.299999
lb      8.600000   0.0  28.400000  27.100000
noise  30.700001  28.4   0.000000  22.299999
wake   29.299999  27.1  22.299999   0.000000
closest pair: la <-> lb at 8.6

la vs lb only: 1233 windows, silhouette 0.011
la-lb centroid gap 8.6 vs within-class spread 36.5 (ratio 0.24)
  ratio < 1 → 

## 4. How much structure did we keep?

If the first 3 components hold only a small share of the variance, a 3-D picture can
look tidy while hiding most of the signal — read this before trusting §3.

In [8]:
evr = pca30.explained_variance_ratio_
fig2 = go.Figure(go.Bar(x=[f"PC{i + 1}" for i in range(len(evr))], y=evr))
fig2.update_layout(
    height=320, title="Explained variance per principal component",
    yaxis_title="share of variance", xaxis_title="component",
)
fig2.show()
print("cumulative:", ", ".join(f"top{k}={evr[:k].sum():.1%}" for k in (3, 5, 10, 30)))

cumulative: top3=26.5%, top5=32.6%, top10=42.2%, top30=61.1%


## 5. Where do the *hold-out* windows land?

The hold-out clips never enter the archive. Projecting them with the **same** scaler +
PCA (no refit) shows whether they fall inside the training clusters or outside them —
which is what a low recall actually looks like in feature space.

In [9]:
if ho is not None and len(ho["features"]) and H is not None:
    HZ = pca30.transform(scaler.transform(H))[:, :3]

    fig3 = go.Figure()
    for label in sorted(df["label"].unique()):
        sub = df[df.label == label]
        fig3.add_trace(go.Scatter3d(
            x=sub[axis_names[0]], y=sub[axis_names[1]], z=sub[axis_names[2]],
            mode="markers", name=f"{label} (train)",
            marker=dict(size=2, opacity=0.25), hoverinfo="skip",
        ))
    fig3.add_trace(go.Scatter3d(
        x=HZ[:, 0], y=HZ[:, 1], z=HZ[:, 2], mode="markers+text",
        name="hold-out",
        marker=dict(size=7, symbol="diamond", color="black", line=dict(width=1, color="white")),
        text=[f"{lbl}: {p}" for lbl, p in zip(ho["labels"], ho["paths"])],
        hovertext=ho["labels"], hovertemplate="%{text}<extra></extra>",
    ))
    fig3.update_layout(
        height=760, title=f"Hold-out windows over the training cloud ({len(ho['labels'])} windows)",
        scene=dict(xaxis_title=axis_names[0], yaxis_title=axis_names[1], zaxis_title=axis_names[2]),
    )
    fig3.write_html(BUILD / "mfcc_3d_holdout.html", include_plotlyjs="cdn")
    print("hold-out windows:", dict(zip(*np.unique(ho["labels"], return_counts=True))))
    fig3.show()

hold-out windows: {np.str_('la'): np.int64(2), np.str_('lb'): np.int64(7), np.str_('noise'): np.int64(2), np.str_('wake'): np.int64(2)}


## 6. Reading the result

Things this notebook can settle in a minute:

* **Is `la` a separate blob from `lb`?** If they interpenetrate even after 1,300-dimensional
  PCA, the model needs better data (variety), not a bigger network.
* **Are augmentations inside their original's neighbourhood?** If `augmented` diamonds sit
  far from `originals` circles of the same colour, an augmentation is distorting the label
  (tempo/pitch too aggressive) and is teaching noise.
* **Do hold-out points sit outside the cloud?** Then it is extrapolation — expect low recall,
  and fix it with data (`shift`/`mix` augmentation, more recordings), not thresholds.
* **Is `noise` a tight blob?** With a median noise clip of 0.46 s, most of the window is
  zero padding, so `noise` tends to look like "silence + a bit of energy" and commands leak
  into it. Compare with `POOL="mean"` to see the effect of dropping timing.

Regenerate with a different `POOL`/`METHOD` at the top of §1 and §2 to compare views.
